# Notebook 02: Preprocessing & Feature Engineering
Data preprocessing, feature engineering, and train-test preparation

## Cell 1 — Imports

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
import pandas as pd
from src.data.data_loader import (
    load_config,
    load_dataset,
    sample_dataset
)
from src.data.preprocessing import (
    basic_preprocessing_pipeline
)
from src.features.feature_engineering import (
    feature_engineering_pipeline
)
from src.models.train_test_split import (
    temporal_train_test_split,
    split_features_target,
    check_class_distribution
)
from src.models.smote_pipeline import (
    apply_smote
)

## Cell 2 — Load Config

In [2]:
config = load_config()
config

{'project': {'name': 'AccidentXAI', 'random_state': 42},
 'paths': {'raw_data': 'data/raw/US_Accidents_March23.csv',
  'processed_data': 'data/processed/processed_accidents.csv'},
 'target': {'column': 'Severity'},
 'temporal': {'date_column': 'Start_Time',
  'train_end_year': 2021,
  'test_year': 2022},
 'sampling': {'enabled': True, 'sample_size': 100000},
 'missing_values': {'numerical_strategy': 'median',
  'categorical_strategy': 'most_frequent'},
 'smote': {'enabled': True, 'random_state': 42},
 'cross_validation': {'folds': 5},
 'models': {'logistic_regression': True,
  'random_forest': True,
  'xgboost': True,
  'lightgbm': True,
  'catboost': True},
 'evaluation': {'scoring': ['f1_macro',
   'f1_weighted',
   'precision_weighted',
   'recall_weighted',
   'roc_auc_ovr']},
 'shap': {'sample_size': 2000}}

## Cell 3 — Load Dataset

In [3]:
df = load_dataset(config)

df.shape


Loading dataset from:
/Users/nurnafisfuad/Desktop/AccidentXAI/data/raw/US_Accidents_March23.csv


Dataset loaded successfully.



(7728394, 24)

## Cell 4 — Sample Dataset

In [4]:
df = sample_dataset(df, config)
df.shape


Sampling 100000 rows...



(100000, 24)

## Cell 5 — Run Preprocessing

In [5]:
df = basic_preprocessing_pipeline(df)
df.head()


Starting preprocessing pipeline...

Converting Start_Time to datetime format...
Removed 85 duplicate rows.

Selected relevant columns.

Dropping high-missing columns:

[]

Missing values handled successfully.

Preprocessing completed successfully.



,Severity,Start_Time,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Weather_Condition,Amenity,Bump,...,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Sunrise_Sunset,State,Start_Lat,Start_Lng
7133276,1,2020-04-17 09:29:30,78.0,81.0,30.13,10.0,13.0,Mostly Cloudy,False,False,...,False,False,False,False,False,True,Day,FL,26.706900,-80.119360
5363845,2,NaT,55.0,88.0,29.83,10.0,9.0,Mostly Cloudy,False,False,...,False,False,False,True,False,False,Day,CA,38.781024,-121.265820
155993,3,2016-08-12 16:45:00,91.0,47.0,29.91,10.0,10.4,Partly Cloudy,False,False,...,False,False,False,False,False,False,Day,GA,33.985249,-84.269348
1861414,3,2019-09-20 15:22:16,67.0,84.0,29.78,10.0,3.0,Cloudy,False,False,...,False,False,False,False,False,False,Day,WA,47.118706,-122.556908
2021359,2,2019-06-03 16:55:43,95.0,16.0,28.53,10.0,6.0,Fair,False,False,...,False,False,False,False,False,False,Day,AZ,33.451355,-111.890343


## Cell 6 — Feature Engineering

In [6]:
df = feature_engineering_pipeline(df)
df.head()


Starting feature engineering pipeline...

Creating temporal features...

Creating rush-hour feature...

Creating night-driving feature...

Simplifying target variable...


Feature engineering completed.



,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Weather_Condition,Amenity,Bump,Crossing,Give_Way,...,Start_Lat,Start_Lng,Year,Month,Day,Hour,Weekday,Rush_Hour,Night_Driving,Severity_Binary
7133276,78.0,81.0,30.13,10.0,13.0,Mostly Cloudy,False,False,False,False,...,26.706900,-80.119360,2020.0,4.0,17.0,9.0,4.0,1,0,0
5363845,55.0,88.0,29.83,10.0,9.0,Mostly Cloudy,False,False,True,False,...,38.781024,-121.265820,NaN,NaN,NaN,NaN,NaN,0,0,0
155993,91.0,47.0,29.91,10.0,10.4,Partly Cloudy,False,False,True,False,...,33.985249,-84.269348,2016.0,8.0,12.0,16.0,4.0,1,0,1
1861414,67.0,84.0,29.78,10.0,3.0,Cloudy,False,False,False,False,...,47.118706,-122.556908,2019.0,9.0,20.0,15.0,4.0,0,0,1
2021359,95.0,16.0,28.53,10.0,6.0,Fair,False,False,False,False,...,33.451355,-111.890343,2019.0,6.0,3.0,16.0,0.0,1,0,0


## Cell 7 — Check Engineered Features

In [7]:
df.columns.tolist()

['Temperature(F)',
 'Humidity(%)',
 'Pressure(in)',
 'Visibility(mi)',
 'Wind_Speed(mph)',
 'Weather_Condition',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Sunrise_Sunset',
 'State',
 'Start_Lat',
 'Start_Lng',
 'Year',
 'Month',
 'Day',
 'Hour',
 'Weekday',
 'Rush_Hour',
 'Night_Driving',
 'Severity_Binary']

## Cell 8 — Temporal Split

In [8]:
train_df, test_df = temporal_train_test_split(
    df,
    train_end_year=2021,
    test_year=2022
)


Performing temporal split...

Train Shape: (71919, 30)
Test Shape: (16318, 30)


## Cell 9 — Feature/Target Split

In [9]:
X_train, X_test, y_train, y_test = split_features_target(
    train_df,
    test_df
)


Separating features and target...

X_train shape: (71919, 29)
X_test shape: (16318, 29)


## Cell 10 — Class Distribution

In [10]:
check_class_distribution(
    y_train,
    y_test
)


Train Class Distribution:

Severity_Binary
0    0.752764
1    0.247236
Name: proportion, dtype: float64

Test Class Distribution:

Severity_Binary
0    0.911386
1    0.088614
Name: proportion, dtype: float64


## Cell 11 — Apply SMOTE

In [11]:
X_train_smote, y_train_smote = apply_smote(
    X_train,
    y_train
)


Applying SMOTE...

Before SMOTE:

Counter({0: 54138, 1: 17781})


ValueError: could not convert string to float: 'Mostly Cloudy'

## Cell 12 — Verify Shapes

In [ ]:
print(f"X_train_smote shape: {X_train_smote.shape}")
print(f"y_train_smote shape: {y_train_smote.shape}")
print(f"\nOriginal X_train shape: {X_train.shape}")
print(f"SMOTE increased samples by: {X_train_smote.shape[0] - X_train.shape[0]} ({((X_train_smote.shape[0] - X_train.shape[0])/X_train.shape[0])*100:.1f}%)")